# Module 2 — Exercise Solutions

---

## Exercise 1: Hyperparameter Impact Analysis — Solution

In [ ]:
# Task 1-2: Calculate training steps

# v1
v1_steps_per_epoch = 2000 // 8  # = 250
v1_total_steps = v1_steps_per_epoch * 3  # = 750
v1_warmup_pct = 75 / v1_total_steps * 100  # = 10.0%

# v2
v2_steps_per_epoch = 2000 // 8  # = 250
v2_total_steps = v2_steps_per_epoch * 2  # = 500
v2_warmup_pct = 50 / v2_total_steps * 100  # = 10.0%

print(f"v1: {v1_total_steps} total steps, {v1_warmup_pct:.1f}% warmup")
print(f"v2: {v2_total_steps} total steps, {v2_warmup_pct:.1f}% warmup")
print()
print("Both use ~10% warmup, which is standard.")
print(f"v1 trains 50% more steps ({v1_total_steps} vs {v2_total_steps}).")

**Predictions:**

**3. Doubling v2 learning rate (5e-5 → 1e-4):**
- Accuracy would likely **stay same or slightly degrade** — the current lr=5e-5
  is already gentle enough for the model to learn. Doubling it risks overshooting
  the optimal loss landscape, especially with only 500 steps.
- Helpfulness would likely **degrade further** — a higher learning rate causes
  the model to move further from the base model's weights, amplifying the
  over-conciseness problem we already saw. The v2 model already learned "too well"
  to be concise; a higher lr would make this worse.

**4. Training v2 for 5 epochs (instead of 2):**
- Main risk: **Overfitting** — with only 2,000 training examples, 5 epochs means
  the model sees each example 5 times. It may memorize the training data rather
  than learning general patterns.
- Metric to watch: **Helpfulness** — an overfit model produces formulaic responses
  that score well on accuracy (memorized facts) but poorly on helpfulness
  (doesn't adapt to the specific question being asked).
  Also watch for validation loss diverging from training loss.

---

## Exercise 2: Analyze Benchmark Results — Solution

In [ ]:
import json

# Load benchmark results
with open("results/benchmark_results_v1.json") as f:
    v1_results = json.load(f)

with open("results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

print(f"v1: {len(v1_results)} results")
print(f"v2: {len(v2_results)} results")

In [ ]:
# Task 2: Compare average response length

for label, results in [("v1", v1_results), ("v2", v2_results)]:
    base_lengths = [len(r["base_output"]) for r in results]
    ft_lengths = [len(r["finetuned_output"]) for r in results]
    
    avg_base = sum(base_lengths) / len(base_lengths)
    avg_ft = sum(ft_lengths) / len(ft_lengths)
    
    print(f"\n{label}:")
    print(f"  Base model avg length: {avg_base:.0f} chars")
    print(f"  Fine-tuned avg length: {avg_ft:.0f} chars")
    print(f"  Change: {avg_ft - avg_base:+.0f} chars ({(avg_ft/avg_base - 1)*100:+.1f}%)")

In [ ]:
# Task 3: For v2, find which questions got shorter

print("v2: Questions that got SHORTER after fine-tuning:")
print(f"{'Question':<55} {'Base':>6} {'FT':>6} {'Delta':>7}")
print("-" * 80)

for r in v2_results:
    base_len = len(r["base_output"])
    ft_len = len(r["finetuned_output"])
    delta = ft_len - base_len
    
    if delta < 0:
        question = r["prompt"][:52] + "..." if len(r["prompt"]) > 55 else r["prompt"]
        print(f"{question:<55} {base_len:>6} {ft_len:>6} {delta:>+7}")

**Answers:**

1. **v2 changes length more dramatically.** The v2 fine-tuning made answers significantly
   shorter because the WikiDoc training data was reformatted to be "concise and focused."

2. **Yes, there's a correlation.** The LangSmith helpfulness evaluator penalized shorter
   answers because they lacked sufficient detail. Shorter answers ≠ better answers when
   patients need comprehensive health information.

3. **Training data directly shapes output style.** When we told GPT-4o-mini to make
   WikiDoc answers "concise and focused," the fine-tuned model learned that brevity was
   the goal. The model faithfully reproduced the conciseness of its training data, even
   when that reduced helpfulness.

**Key lesson:** Your training data's style IS your model's style after fine-tuning.

---

## Exercise 3: LoRA Rank Experiment — Solution

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate peft trl datasets

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

In [ ]:
# r=4 LoRA configuration
lora_config_r4 = LoraConfig(
    r=4,
    lora_alpha=8,        # alpha = 2*r is a common heuristic
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config_r4)
peft_model.print_trainable_parameters()

# Expected output: ~3.5M trainable params (roughly 25% of r=16's ~14M)
# This makes sense: LoRA adds r × (d_in + d_out) params per layer,
# so halving r halves the LoRA params, and 4/16 = 25%.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("jeev1992/wikidoc-healthassist", split="train")
dataset = dataset.shuffle(seed=42)
train_dataset = dataset.select(range(2000))
eval_dataset = dataset.select(range(2000, 2100))

training_args = SFTConfig(
    output_dir="./lora-r4-experiment",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch size = 8
    learning_rate=5e-5,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=250,
    bf16=True,
    max_seq_length=512,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Benchmark with the same 10 questions
BENCHMARK_PROMPTS = [
    "What are the common symptoms of type 2 diabetes?",
    "How does hypertension affect the cardiovascular system?",
    "What are the early warning signs of a stroke?",
    "How does metformin work for diabetes management?",
    "What lifestyle changes help manage high cholesterol?",
    "What are the risk factors for developing osteoporosis?",
    "How does asthma affect the respiratory system?",
    "What preventive measures reduce the risk of heart disease?",
    "What are the treatment options for rheumatoid arthritis?",
    "How does chronic kidney disease progress through its stages?",
]

system_prompt = (
    "You are a helpful healthcare assistant. Provide accurate, "
    "evidence-based health information. Always recommend consulting "
    "healthcare professionals for personalized medical advice."
)

print("Generating r=4 outputs...\n")
for i, prompt in enumerate(BENCHMARK_PROMPTS[:3]):  # first 3 for comparison
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(peft_model.device)
    
    with torch.no_grad():
        outputs = peft_model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q{i+1}: {prompt}")
    print(f"A:  {response[:300]}...\n")

**Expected Findings:**

1. **Trainable parameters:** r=4 has ~3.5M params vs r=16 had ~14M (roughly 1/4)
2. **Training loss:** r=4 likely converges to a slightly higher final loss because
   it has less capacity to adapt. The difference is small for a 1.5B model.
3. **Output quality:** For most healthcare Q&A, r=4 and r=16 produce very similar
   outputs. The 1.5B base model already knows most of the medical knowledge — LoRA
   just adjusts the output style. r=4 is sufficient for style transfer.
4. **For 2,000 training examples:** r=4 is actually a reasonable choice. With only
   2,000 examples, r=16 might be overparameterized. A lower rank acts as implicit
   regularization, potentially reducing overfitting.

---

## Exercise 4: Training Data Ablation — Solution

In [ ]:
# Data ablation: compare 200, 500, and 2000 examples
# NOTE: Run these sequentially to avoid OOM. Clear cache between runs.

from datasets import load_dataset

dataset = load_dataset("jeev1992/wikidoc-healthassist", split="train")
dataset = dataset.shuffle(seed=42)

test_questions = [
    "What are the early warning signs of a stroke?",
    "How does metformin work for diabetes management?",
    "What lifestyle changes help manage high cholesterol?",
]

ablation_results = {}

for size in [200, 500]:  # 2000 is already done from class
    print(f"\n{'='*60}")
    print(f"Training with {size} examples...")
    print(f"{'='*60}")
    
    # Reload model fresh
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules="all-linear", task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    
    train_subset = dataset.select(range(size))
    
    args = SFTConfig(
        output_dir=f"./ablation-{size}",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        warmup_steps=min(50, size // 8),  # Scale warmup with data size
        weight_decay=0.01,
        logging_steps=10,
        bf16=True,
        max_seq_length=512,
        report_to="none",
        seed=42,
    )
    
    trainer = SFTTrainer(
        model=peft_model, args=args,
        train_dataset=train_subset,
        processing_class=tokenizer,
    )
    trainer.train()
    
    # Generate outputs
    ablation_results[size] = []
    for q in test_questions:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(peft_model.device)
        with torch.no_grad():
            outputs = peft_model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        ablation_results[size].append({"question": q, "response": response})
        print(f"\n  Q: {q}")
        print(f"  A: {response[:200]}...")
    
    # Cleanup
    del peft_model, model, trainer
    torch.cuda.empty_cache()

**Expected Findings:**

1. **200 examples:** Safety disclaimers may be inconsistent. The model learns some
   patterns but hasn't seen enough variety to generalize well. Output quality: ~3/5.
   Responses may be incomplete or lack the structured format.

2. **500 examples:** Safety disclaimers more consistent. The model produces reasonable
   answers with better structure. Output quality: ~3.5/5. Main gap vs 2,000 is
   less variety in phrasing and occasional format inconsistency.

3. **2,000 examples (from class):** Safety disclaimers at 99.4% (from our audit).
   Output quality: ~4/5. Consistent formatting, but over-concise (the helpfulness
   regression issue).

4. **Diminishing returns** start at approximately 500-1,000 examples because:
   - The model already has medical knowledge from pretraining
   - LoRA is only adjusting output STYLE, not teaching new facts
   - After ~500 examples, the model has seen enough style patterns to generalize
   - Going from 500→2,000 mostly reinforces consistency, not quality

5. **Minimum recommendation:** ~500 examples for a proof of concept, ~1,000 for
   production-quality consistency. Below 200, the model doesn't reliably adopt the
   target style.